<a href="https://colab.research.google.com/github/shuvad23/Foundation-Introduction-to-LangChain---Python/blob/main/1_3_prompting_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
print(GEMINI_API_KEY[:6], "****")  # sanity check

TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")
print(TAVILY_API_KEY[:6], "****")

AIzaSy ****
tvly-d ****


In [4]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

print("Available Gemini models that support generateContent:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Gemini models that support generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


## Tool definition

In [5]:
! pip install langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00


In [6]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Returns the square root of the input."""
    return x**0.5
@tool("square_root")
def tool1(x: float) -> float:
    """Returns the square root of the input."""
    return x**0.5
@tool("square_root", description="Returns the square root of the input.")
def tool2(x: float) -> float:
    return x**0.5


In [7]:
tool1.invoke({"x":169})

13.0

## Adding to agents

In [8]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash-lite",
    temperature=0.5,
    max_output_tokens=1024,
    google_api_key=GEMINI_API_KEY,
    top_k=40,
    top_p=0.95
)

system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."

agent = create_agent(
    model = llm,
    tools = [square_root, tool1, tool2],
    system_prompt=system_prompt
)

In [9]:
from langchain.messages import HumanMessage

question = HumanMessage(content = "What is the square root of 437?")
response = agent.invoke(
    {'messages':[question]}
)

print(response['messages'][-1].content)

The square root of 437 is approximately 20.90.


In [10]:
from pprint import pprint
pprint(response['messages'])

[HumanMessage(content='What is the square root of 437?', additional_kwargs={}, response_metadata={}, id='1b64a911-3295-4ef3-8f98-9037ffa1dc74'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'square_root', 'arguments': '{"x": 437}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c941e-1088-74b3-ae3e-0fc8df7d90a5-0', tool_calls=[{'name': 'square_root', 'args': {'x': 437}, 'id': 'ee35cf90-5082-42fe-8e94-1441a90f7f7f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 17, 'total_tokens': 89, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='20.904544960366874', name='square_root', id='4a98e72c-1e09-4ce5-bad8-b2917f0f84eb', tool_call_id='ee35cf90-5082-42fe-8e94-1441a90f7f7f'),
 AIMessage(content='The square root of 437 is approximately 20.90.', additional_kwargs={}, response_metadata={'finish_rea

In [11]:
print(response['messages'][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 437}, 'id': 'ee35cf90-5082-42fe-8e94-1441a90f7f7f', 'type': 'tool_call'}]


In [12]:
! pip install arxiv


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.0 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=87af9f8a4b5c3dd6ee01b4bcb46d2b0bf47c3bff850534560a22d0008ae8d992
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


## Mini Research Agent Architecture

In [13]:
import arxiv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

In [34]:
llm = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    temperature=0.5,
    max_output_tokens=1024,
    google_api_key=GEMINI_API_KEY,
    top_k=40,
    top_p=0.95
)

In [35]:
# -----------------------------
# Tool 1: ArXiv Search
# -----------------------------

@tool("Search_Arxiv", description="Search research papers on ArXiv by topic")
def Search_Arxiv(query: str) -> str:
    search = arxiv.Search(
        query = query,
        max_results=3,
        sort_by = arxiv.SortCriterion.Relevance
    )
    results = []
    for paper in search.results():
        results.append(
            f"Title: {paper.title}\n"
            f"Authors: {", ".join(a.name for a in paper.authors)}\n"
            f"Summary: {paper.summary}\n"
            f"Link: {paper.entry_id}\n"
        )
    return "\n\n".join(results)

arxiv_tool = Search_Arxiv

In [42]:
# -----------------------------
# Agent
# -----------------------------
agent = create_agent(
    llm,
    tools=[arxiv_tool],
    system_prompt="""
    You are a research assistant.
    ALWAYS use Search_Arxiv when user asks about research papers.
    """
)

In [47]:
# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    # The agent expects a dictionary with a 'messages' key, containing a HumanMessage
    from langchain.messages import HumanMessage
    question_message = HumanMessage(content="Latest research on large language models")
    answer = agent.invoke({'messages': [question_message]})
    print("\nFinal Answer Metadata:\n", answer['messages'][-1].response_metadata)
    print("\nFinal Answer Content:\n", answer['messages'][-1].content)
    print("\nFinal Answer Tool Calls:\n", answer['messages'][-1].tool_calls)

/tmp/ipython-input-2336108919.py:34: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for paper in search.results():



Final Answer Metadata:
 {'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}

Final Answer Content:
 [{'type': 'text', 'text': "Here is some of the latest research on large language models:\n\n*   **Title**: Enhancing Human-Like Responses in Large Language Models\n    **Authors**: Ethem Yağız Çalık, Talha Rüzgar Akkuş\n    **Summary**: This paper explores advancements in making large language models (LLMs) more human-like, focusing on techniques that enhance natural language understanding, conversational coherence, and emotional intelligence. The study evaluates various approaches, including fine-tuning with diverse datasets, incorporating psychological principles, and designing models that better mimic human reasoning patterns.\n    **Link**: http://arxiv.org/abs/2501.05032v2\n\n*   **Title**: Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality\n    **Authors**: Yiming Ai, Zhiw

In [48]:
! pip install tavily-python

In [53]:
# import os
# from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from tavily import TavilyClient

# load_dotenv()

# Keys
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# LLM
# llm = ChatGoogleGenerativeAI(
#     model="gemini-1.5-flash",
#     temperature=0.3
# )

# Tavily Client
tavily = TavilyClient(api_key=TAVILY_API_KEY)

# -----------------------------
# Tool: Tavily Search
# -----------------------------
@tool("Web_Search", description="Search latest research and info from internet")
def web_search(query: str) -> str:
    result = tavily.search(query=query, max_results=5)

    output = ""
    for r in result["results"]:
        output += f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content'][:300]}...\n\n"
    return output

# -----------------------------
# Agent
# -----------------------------
agent = create_agent(
    llm,
    tools=[web_search],
    system_prompt="""
    You are an AI research assistant.
    ALWAYS search the web when user asks about research, news, or latest info.
    """
)

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    question = HumanMessage(
        content="Latest research on large language models"
    )

    answer = agent.invoke({"messages": [question]})

    print("\nFinal Answer:\n", answer["messages"][-1].content[0]['text'])


Final Answer:
 Here's some of the latest research on large language models:

*   **A new way to increase the capabilities of large language models** (MIT-IBM Watson AI Lab): Researchers developed an expressive architecture that improves state tracking and sequential reasoning in LLMs.
*   **A large-scale randomized study of large language model feedback** (Nature): This study demonstrated that LLMs can be effective critics, providing detailed and constructive feedback.
*   **Large Language Models Reveal the Neural Tracking of Linguistic** (bioRxiv): This research suggests that LLMs capture long-range contextual structure in natural language and align with human cognitive processes.
*   **Latest Large Language Model News: What's Happening Now?** (The Detroit Bureau): This article discusses how LLMs are becoming smarter, faster, and more capable.
*   **Tracing the thoughts of a large language model** (Anthropic): Anthropic has released papers detailing progress on a "microscope" to anal